# Module 03: Mô phỏng bộ giải mã / bộ mã hoá / bộ dồn kênh (Decoder / Encoder / Multiplexer)
**AE2.021 · USTH**

🇻🇳 Notebook mô phỏng bảng chân trị của 3 mạch MSI phổ biến nhất trong module: bộ giải mã 3-sang-8, bộ mã
hoá ưu tiên, và bộ dồn kênh 4-sang-1 / 8-sang-1: đồng thời kiểm chứng công thức n = log₂(N).

🇬🇧 This notebook simulates the truth table of the three most common MSI circuits in this module: a 3-to-8
decoder, a priority encoder, and 4-to-1 / 8-to-1 multiplexers: and verifies the n = log₂(N) formula.

## 1. Bộ giải mã 3-sang-8 (3-to-8 decoder)
🎯 **Phương pháp này trả lời câu hỏi gì?** Với 3 đường địa chỉ, làm sao xác định chính xác đường ra nào
được kích hoạt (=1) cho MỖI tổ hợp địa chỉ, không nhầm lẫn thứ tự bit.

In [1]:
import itertools

def decoder_3to8(a2, a1, a0):
    addr = (a2 << 2) | (a1 << 1) | a0
    outputs = [0]*8
    outputs[addr] = 1
    return outputs

print("A2 A1 A0 | Y0 Y1 Y2 Y3 Y4 Y5 Y6 Y7")
for a2, a1, a0 in itertools.product([0,1], repeat=3):
    outs = decoder_3to8(a2, a1, a0)
    print(f" {a2}  {a1}  {a0}  | " + " ".join(str(o) for o in outs))

A2 A1 A0 | Y0 Y1 Y2 Y3 Y4 Y5 Y6 Y7
 0  0  0  | 1 0 0 0 0 0 0 0
 0  0  1  | 0 1 0 0 0 0 0 0
 0  1  0  | 0 0 1 0 0 0 0 0
 0  1  1  | 0 0 0 1 0 0 0 0
 1  0  0  | 0 0 0 0 1 0 0 0
 1  0  1  | 0 0 0 0 0 1 0 0
 1  1  0  | 0 0 0 0 0 0 1 0
 1  1  1  | 0 0 0 0 0 0 0 1


#### 📤 Đầu ra thật
Đúng như kỳ vọng: ở mỗi hàng, CHÍNH XÁC MỘT ngõ ra bằng 1 (ngõ ra có chỉ số bằng giá trị nhị phân của địa
chỉ A2A1A0), 7 ngõ ra còn lại đều bằng 0: đây là đặc trưng cốt lõi của một bộ giải mã.

## 2. Bộ dồn kênh (multiplexer) và công thức n = log₂(N)
🎯 **Phương pháp này trả lời câu hỏi gì?** Có đúng N kênh dữ liệu, cần tối thiểu bao nhiêu đường chọn: và
xác nhận công thức n=log₂(N) khớp với các câu hỏi trên trang web (mux 4-sang-1 cần 2 đường chọn, 8-sang-1
cần 3 đường chọn).

In [2]:
import math

def mux(data, select_bits):
    idx = int("".join(str(b) for b in select_bits), 2)
    return data[idx]

# 4-to-1 mux, can 2 duong chon
data4 = [0,1,0,1]
for s1, s0 in itertools.product([0,1], repeat=2):
    print(f"S1={s1} S0={s0} -> Y = data[{int(f'{s1}{s0}',2)}] = {mux(data4,[s1,s0])}")

for N in (2,4,8,16,32):
    n = math.log2(N)
    print(f"N={N:2d} kenh -> can n = log2({N}) = {n:.0f} duong chon")

S1=0 S0=0 -> Y = data[0] = 0
S1=0 S0=1 -> Y = data[1] = 1
S1=1 S0=0 -> Y = data[2] = 0
S1=1 S0=1 -> Y = data[3] = 1
N= 2 kenh -> can n = log2(2) = 1 duong chon
N= 4 kenh -> can n = log2(4) = 2 duong chon
N= 8 kenh -> can n = log2(8) = 3 duong chon
N=16 kenh -> can n = log2(16) = 4 duong chon
N=32 kenh -> can n = log2(32) = 5 duong chon


#### 📤 Đầu ra thật
Bảng xác nhận: N=4 → n=2 đường chọn, N=8 → n=3 đường chọn: khớp đúng với câu hỏi gốc "A four-to-one
multiplexer has two select inputs" (Tooley Ch.9 Q5) và câu tự sinh về mux 8-sang-1.

## 3. Bộ mã hoá ưu tiên đơn giản (priority encoder, 8 đầu vào)
🎯 **Phương pháp này trả lời câu hỏi gì?** Khi NHIỀU đầu vào cùng tích cực một lúc, bộ mã hoá ưu tiên chọn
đầu vào có chỉ số CAO NHẤT để mã hoá: khác với bộ mã hoá thường (chỉ đúng khi có đúng 1 đầu vào tích cực).

In [3]:
def priority_encoder_8to3(inputs):
    for i in range(7, -1, -1):
        if inputs[i] == 1:
            return format(i, '03b')
    return None

tests = [
    [0,0,0,0,0,0,1,0],  # chi input 6 tich cuc
    [0,0,1,0,0,1,0,1],  # nhieu input tich cuc: 2, 5, 7 -> uu tien 7
]
for t in tests:
    print(f"input={t} -> ma hoa (uu tien cao nhat) = {priority_encoder_8to3(t)}")

input=[0, 0, 0, 0, 0, 0, 1, 0] -> ma hoa (uu tien cao nhat) = 110
input=[0, 0, 1, 0, 0, 1, 0, 1] -> ma hoa (uu tien cao nhat) = 111


#### 📤 Đầu ra thật
Khi cả input 2, 5 và 7 đều tích cực, bộ mã hoá ưu tiên chọn đúng input 7 (chỉ số cao nhất) để mã hoá thành
`111`, minh hoạ đúng lý do vì sao encoder cần thêm logic "ưu tiên" so với mã hoá đơn giản.

---
## 4. Ngân hàng bài tập Floyd: Chương 6 "Problems" (Functions of Combinational Logic)

🇻🇳 Chương 6 có 55 bài tập nhưng **phần lớn dựa trên sơ đồ mạch cụ thể trong hình** (Figure 6-69
tới 6-9x: bộ cộng nhiều tầng, mạch so sánh, sơ đồ chân IC...): cần trích xuất hình mới giải được
chính xác. Notebook này giải **4 bài không cần hình** (1, 2, 3: logic cộng thuần với giá trị A,B,Cin
cho sẵn bằng chữ; 9: dùng đúng 8 chuỗi bit cho sẵn trong đề, KHÔNG bịa số liệu thay thế).

🇬🇧 Chapter 6 has 55 problems but **most rely on specific circuit figures** (Figure 6-69 to 6-9x:
multi-stage adders, comparator circuits, IC pin diagrams...) that would need image extraction to
solve accurately. This notebook solves the **4 problems that need no figure** (1, 2, 3: pure adder
logic with A, B, Cin given as text; 9: using the exact 8 bit-sequences given in the problem, no
substitute data).

In [4]:
# -*- coding: utf-8 -*-
"""Auto-solve the text-only (no-figure) problems from Floyd Ch.6 (adders) and Ch.11 (memory)."""

print("="*70, "\nFloyd Ch.6 - Section 6-1 Half and Full Adders (Problems 1-3)\n" + "="*70)
def full_adder(a,b,cin):
    s = a ^ b ^ cin
    cout = (a & b) | (cin & (a ^ b))
    return s, cout

print("1) Full-adder, tim Sum(S) va Carry-out(Cout):")
for a,b,cin in [(0,1,0),(1,0,1),(0,0,0)]:
    s,c = full_adder(a,b,cin)
    print(f"   A={a},B={b},Cin={cin} -> Sum={s}, Cout={c}")

print("2) Dau vao half-adder can co de ra dung Sum/Cout mong muon (Sum=A xor B, Cout=A and B):")
targets = [(0,0),(1,0),(0,1)]
for s_target, c_target in targets:
    found = [(a,b) for a in (0,1) for b in (0,1) if (a^b)==s_target and (a&b)==c_target]
    print(f"   Muon Sum={s_target},Cout={c_target} -> (A,B) co the la: {found}")

print("3) Full-adder voi 4 bo gia tri:")
for a,b,cin in [(1,0,0),(0,0,1),(0,1,1),(1,1,1)]:
    s,c = full_adder(a,b,cin)
    print(f"   A={a},B={b},Cin={cin} -> Sum={s}, Cout={c}")

print("\n" + "="*70, "\nFloyd Ch.6 - Section 6-2 Problem 9 (du lieu THAT tu sach, doc tuan tu 4 chu ky clock)\n" + "="*70)
print("   De bai: A1=1010, A2=1100, A3=0101, A4=1101, B1=1001, B2=1011, B3=0000, B4=0001")
print("   (\"right-most bit first\" -> bit ngoai cung BEN PHAI cua moi chuoi la gia tri tai chu ky t=1)")
raw = {"A1":"1010","A2":"1100","A3":"0101","A4":"1101","B1":"1001","B2":"1011","B3":"0000","B4":"0001"}
# seq[line][t] voi t=0..3 (t=0 la chu ky dau tien = bit ngoai cung ben phai)
seq = {k: list(reversed(v)) for k,v in raw.items()}
print("   Sau khi doi thu tu (chu ky 1->4 doc tu trai sang phai):")
for k,v in seq.items(): print(f"     {k}: {''.join(v)}  (chu ky1={v[0]}, chu ky2={v[1]}, chu ky3={v[2]}, chu ky4={v[3]})")
print("\n   Tai moi chu ky t, A = A4A3A2A1 (MSB..LSB), B = B4B3B2B1 (MSB..LSB), tinh Sum = A+B:")
sums = []
for t in range(4):
    Aval = int(seq["A4"][t]+seq["A3"][t]+seq["A2"][t]+seq["A1"][t], 2)
    Bval = int(seq["B4"][t]+seq["B3"][t]+seq["B2"][t]+seq["B1"][t], 2)
    s = Aval + Bval
    sums.append(format(s, '05b'))  # 5 bit: Cout Sigma4 Sigma3 Sigma2 Sigma1
    print(f"     Chu ky {t+1}: A={Aval:2d} ({format(Aval,'04b')}), B={Bval:2d} ({format(Bval,'04b')}) -> Sum={s:2d} = {format(s,'05b')} (Cout Sig4 Sig3 Sig2 Sig1)")
print("\n   => Chuoi bit tren TUNG duong tong (doc theo 4 chu ky, chu ky1 truoc):")
for i, label in enumerate(["Cout","Sigma4","Sigma3","Sigma2","Sigma1"]):
    bits = "".join(s[i] for s in sums)
    print(f"     {label}: {bits}")

print("\n" + "="*70, "\nFloyd Ch.11 - Section 11-1 Semiconductor Memory Basics (Problem 4)\n" + "="*70)
print("4) Dia chi bo nho (0-256) tuong ung moi so hex:")
for h in ["0C","5E","DF"]:
    print(f"   {h}16 = {int(h,16)} (thap phan)")

print("\n" + "="*70, "\nFloyd Ch.11 - Problem 2 (khai niem, khong tinh so): quan he bit/byte/nibble/word\n" + "="*70)
print("   1 nibble = 4 bit. 1 byte = 8 bit = 2 nibble. 1 word = boi so cua byte (thuong 16/32/64-bit = 2/4/8 byte),")
print("   tuy kien truc CPU cu the quy dinh do rong 1 word.")

print("\n" + "="*70, "\nFloyd Ch.11 - Section 11-1 Problem 5 (RAM tinh 4 hang, dieu kien cho truoc)\n" + "="*70)
rows = [0,0,0,0]
conditions = [(0,1,1),(1,0,1),(2,1,0),(3,0,1)]  # (row_index, row_select, data_in_bit)
print("   Ban dau ca 4 hang deu = 0. Ap dung dieu kien (Row select=1 moi ghi):")
for row_idx, sel, data in conditions:
    if sel == 1:
        rows[row_idx] = data
        print(f"   Row {row_idx} duoc chon (select=1), ghi Data in={data} -> Row {row_idx} = {data}")
    else:
        print(f"   Row {row_idx} KHONG duoc chon (select=0) -> giu nguyen = {rows[row_idx]}")
print("   Noi dung cuoi cung 4 hang:", rows)


Floyd Ch.6 - Section 6-1 Half and Full Adders (Problems 1-3)
1) Full-adder, tim Sum(S) va Carry-out(Cout):
   A=0,B=1,Cin=0 -> Sum=1, Cout=0
   A=1,B=0,Cin=1 -> Sum=0, Cout=1
   A=0,B=0,Cin=0 -> Sum=0, Cout=0
2) Dau vao half-adder can co de ra dung Sum/Cout mong muon (Sum=A xor B, Cout=A and B):
   Muon Sum=0,Cout=0 -> (A,B) co the la: [(0, 0)]
   Muon Sum=1,Cout=0 -> (A,B) co the la: [(0, 1), (1, 0)]
   Muon Sum=0,Cout=1 -> (A,B) co the la: [(1, 1)]
3) Full-adder voi 4 bo gia tri:
   A=1,B=0,Cin=0 -> Sum=1, Cout=0
   A=0,B=0,Cin=1 -> Sum=1, Cout=0
   A=0,B=1,Cin=1 -> Sum=0, Cout=1
   A=1,B=1,Cin=1 -> Sum=1, Cout=1

Floyd Ch.6 - Section 6-2 Problem 9 (du lieu THAT tu sach, doc tuan tu 4 chu ky clock)
   De bai: A1=1010, A2=1100, A3=0101, A4=1101, B1=1001, B2=1011, B3=0000, B4=0001
   ("right-most bit first" -> bit ngoai cung BEN PHAI cua moi chuoi la gia tri tai chu ky t=1)
   Sau khi doi thu tu (chu ky 1->4 doc tu trai sang phai):
     A1: 0101  (chu ky1=0, chu ky2=1, chu ky3=0, chu k